# 06 — Checkpointing and Resume

**Learning objective:** pause an execution, identify it with a stable thread ID, restore persisted state, and resume without replaying completed nodes.

A temporary pipeline loses its place when execution stops. A checkpointer records graph state and the next scheduled node, which matters for long-running work, human approval, tool outages, process restarts, and multi-step business workflows. Here persistence is in memory so no external infrastructure is required.

## Mental model and topology

```mermaid
flowchart TD
    accTitle: Checkpointed workflow
    accDescr: Preparation, research, and review are checkpointed before execution pauses at a protected action and later resumes.

    start_node([Start]) --> prepare[Prepare]
    prepare --> research[Research]
    research --> review[Build review packet]
    review --> pause_node{{Pause before action}}
    pause_node -. same thread ID .-> protected_action[Protected action]
    protected_action --> finish([End])
```

## State, nodes, and control policy

The state carries `request`, `idempotency_key`, evidence, review data, and `visits`. Nodes compute those updates. The compiled graph pauses **before** `protected_action`; the checkpointer and `thread_id` preserve where execution should continue. The append reducer on `visits` makes replay visible.

In [1]:
from graph_engineering.persistence import build_checkpoint_graph, thread_config

graph, ledger, memory = build_checkpoint_graph()
config = thread_config("lesson-06-run")

In [2]:
paused = graph.invoke(
    {"request": "publish the research brief",
     "idempotency_key": "publish-brief-001",
     "visits": []},
    config,
)
print("Next node:", graph.get_state(config).next)
print("Visited before pause:", paused["visits"])
print("Action executions:", ledger.execution_count)

Next node: ('protected_action',)
Visited before pause: ['prepare', 'research', 'review']
Action executions: 0


In [3]:
resumed = graph.invoke(None, config)
print("Visited after resume:", resumed["visits"])
print("Termination:", resumed["termination_reason"])
print("Action executions:", ledger.execution_count)

Visited after resume: ['prepare', 'research', 'review', 'protected_action', 'finish']
Termination: completed
Action executions: 1


## Failure case: checkpoint does not mean safe side effects

`charge_customer → crash → resume → charge_customer again` is possible when the external action is not idempotent. A checkpoint protects workflow progress, not the semantics of an external system. The simulated ledger below accepts an idempotency key and executes a logical action at most once, even if a new run submits the same request.

In [4]:
replay_config = thread_config("lesson-06-replayed-request")
graph.invoke(
    {"request": "publish the research brief",
     "idempotency_key": "publish-brief-001",
     "visits": []},
    replay_config,
)
replayed = graph.invoke(None, replay_config)
print("Replay status:", replayed["action_status"])
print("Total action executions:", ledger.execution_count)

Replay status: already_executed
Total action executions: 1


## What to modify

Try a new `thread_id`, move the interrupt boundary, or replace the ledger with a fake service that records keys. Replaying a completed thread is different from starting a new thread with the same business operation—both need a deliberate policy.

**Next:** [07 — Human-in-the-Loop](07_human_in_the_loop.ipynb) uses persisted state with an explicit human decision.